# Get Satellite Images from French Intersection Dataset

In [2]:
import math
import requests
from PIL import Image
from io import BytesIO
import pandas as pd
import os
import json

#### functions for image download

In [3]:
api_key = "qR5z0FXl7vgm9kk146HC"

In [4]:
def deg2num(lat, lon, zoom):
    """
    Umrechnung Lat/Lon → XYZ Tile‑Koordinaten
    """
    lat_rad = math.radians(lat)
    n = 2.0 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.log(math.tan(lat_rad) +
                (1 / math.cos(lat_rad))) / math.pi) / 2.0 * n)
    return x, y

def fetch_tile(map_id, zoom, x, y, api_key):
    """
    Lädt eine einzelne Tile‑Kachel
    """
    url = f"https://api.maptiler.com/maps/{map_id}/256/{zoom}/{x}/{y}.png?key={api_key}"
    r = requests.get(url)
    r.raise_for_status()
    return Image.open(BytesIO(r.content))

def build_map(lat, lon, zoom, tiles_radius, api_key, map_id="satellite"):
    """
    Erstellt aus mehreren Tiles ein großes Bild
    """
    center_x, center_y = deg2num(lat, lon, zoom)

    size = 256 * (tiles_radius * 2 + 1)
    canvas = Image.new("RGB", (size, size))

    for dx in range(-tiles_radius, tiles_radius + 1):
        for dy in range(-tiles_radius, tiles_radius + 1):
            x = center_x + dx
            y = center_y + dy
            tile = fetch_tile(map_id, zoom, x, y, api_key)
            px = (dx + tiles_radius) * 256
            py = (dy + tiles_radius) * 256
            canvas.paste(tile, (px, py))

    return canvas


# Parameters

zoom = 21
tiles_radius = 2

## Get Images of Intersections

In [5]:
# Filter for relevant intersections (Caen)

input_geojson = "intersections-Calvados/intersections-14.geojson"
output_geojson = "intersections-Calvados/intersections-14118.geojson"


with open(input_geojson, "r", encoding="utf-8") as f:
    data = json.load(f)

filtered_features = [
    feature for feature in data["features"]
    if feature.get("properties", {}).get("citycode") == "14118"
]

filtered_geojson = {
    "type": "FeatureCollection",
    "features": filtered_features
}


with open(output_geojson, "w", encoding="utf-8") as f:
    json.dump(filtered_geojson, f, ensure_ascii=False, indent=2)

In [6]:
# import data from GeoJSON
fp_intersections = "intersections-Calvados/intersections-14118.geojson"

with open(fp_intersections, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

features = geojson_data["features"]

df_intersection = pd.DataFrame([feature["properties"] for feature in features])

df_intersection["longitude"] = [feature["geometry"]["coordinates"][0] for feature in features]
df_intersection["latitude"] = [feature["geometry"]["coordinates"][1] for feature in features]

df_intersection = df_intersection[~df_intersection["name"].str.contains("Bus", case=False, na=False)]

print(df_intersection.columns)
print(f"Anzahl Einträge: {len(df_intersection)}")

df_intersection.head()

Index(['name', 'context', 'citycode', 'depcode', 'longitude', 'latitude'], dtype='object')
Anzahl Einträge: 2763


,name,context,citycode,depcode,longitude,latitude
0,Acqueduc Vendeuvre / Quai François Mitterrand,"Caen, Calvados",14118,14,-0.355980,49.183508
1,Allée Baudelaire / Rue Constant Forget,"Caen, Calvados",14118,14,-0.388655,49.169757
2,Allée Cécile de Normandie / Avenue Georges Cle...,"Caen, Calvados",14118,14,-0.349447,49.188695
3,Allée Charles Gassion / Rue Germaine Tillion,"Caen, Calvados",14118,14,-0.345283,49.163297
4,Allée d'Auderville / Allée de Jobourg,"Caen, Calvados",14118,14,-0.402072,49.192026


In [39]:
df_test = df_intersection
df_test = df_test.sample(n=300)

df_test.head()

,name,context,citycode,depcode,longitude,latitude
4975,Rue Fernand Léger / Rue du Chemin Vert,"Caen, Calvados",14118,14,-0.387000,49.186564
3331,Caen Tour Leroy → Caen Parking Parc Expo / Ave...,"Caen, Calvados",14118,14,-0.371127,49.177612
175,Avenue de la Côte de Nacre D7 / Boulevard Maré...,"Caen, Calvados",14118,14,-0.363387,49.210953
3688,Promenade Charles Lamusse / Route d'Harcourt D562,"Caen, Calvados",14118,14,-0.364817,49.167069
4049,Rue de Bayeux / Place de l'Ancienne Boucherie,"Caen, Calvados",14118,14,-0.374577,49.182383


In [40]:
save_path = "img_intersections"
if not os.path.exists(save_path):
    os.makedirs(save_path)

i = 0
for i in range(len(df_test)):
    img_name = f"img_{i+1}"

    # get coords
    dp = df_test.iloc[i]
    lat, lon = dp["latitude"], dp["longitude"]


    # Get Images
    map_img = build_map(lat, lon, zoom, tiles_radius, api_key)
    map_img.save(f"{save_path}/{img_name}.png")
    print(f"{img_name} saved:     lat: {lat} N, lon: {lon} E        {i+1}/{len(df_test)}")

    i +=1

img_1 saved:     lat: 49.186563593 N, lon: -0.38699953 E        1/300
img_2 saved:     lat: 49.177612494 N, lon: -0.371126948 E        2/300
img_3 saved:     lat: 49.210952536 N, lon: -0.363387348 E        3/300
img_4 saved:     lat: 49.167069076 N, lon: -0.36481708 E        4/300
img_5 saved:     lat: 49.182383369 N, lon: -0.374576657 E        5/300
img_6 saved:     lat: 49.186340845 N, lon: -0.376027232 E        6/300
img_7 saved:     lat: 49.200531158 N, lon: -0.387272006 E        7/300
img_8 saved:     lat: 49.188061713 N, lon: -0.36261409 E        8/300
img_9 saved:     lat: 49.186434334 N, lon: -0.366117353 E        9/300
img_10 saved:     lat: 49.180319005 N, lon: -0.401854215 E        10/300
img_11 saved:     lat: 49.192408364 N, lon: -0.395360722 E        11/300
img_12 saved:     lat: 49.166878996 N, lon: -0.355790953 E        12/300
img_13 saved:     lat: 49.174465355 N, lon: -0.389486551 E        13/300
img_14 saved:     lat: 49.183890069 N, lon: -0.369501253 E        14/300

### coordinates of currently used images

img_1 saved:     lat: 49.186563593 N, lon: -0.38699953 E        1/300
img_2 saved:     lat: 49.177612494 N, lon: -0.371126948 E        2/300
img_3 saved:     lat: 49.210952536 N, lon: -0.363387348 E        3/300
img_4 saved:     lat: 49.167069076 N, lon: -0.36481708 E        4/300
img_5 saved:     lat: 49.182383369 N, lon: -0.374576657 E        5/300
img_6 saved:     lat: 49.186340845 N, lon: -0.376027232 E        6/300
img_7 saved:     lat: 49.200531158 N, lon: -0.387272006 E        7/300
img_8 saved:     lat: 49.188061713 N, lon: -0.36261409 E        8/300
img_9 saved:     lat: 49.186434334 N, lon: -0.366117353 E        9/300
img_10 saved:     lat: 49.180319005 N, lon: -0.401854215 E        10/300
img_11 saved:     lat: 49.192408364 N, lon: -0.395360722 E        11/300
img_12 saved:     lat: 49.166878996 N, lon: -0.355790953 E        12/300
img_13 saved:     lat: 49.174465355 N, lon: -0.389486551 E        13/300
img_14 saved:     lat: 49.183890069 N, lon: -0.369501253 E        14/300
img_15 saved:     lat: 49.182110586 N, lon: -0.380996338 E        15/300
img_16 saved:     lat: 49.178988306 N, lon: -0.35648802 E        16/300
img_17 saved:     lat: 49.2082329 N, lon: -0.359484508 E        17/300
img_18 saved:     lat: 49.172325191 N, lon: -0.381209139 E        18/300
img_19 saved:     lat: 49.1831298 N, lon: -0.355958172 E        19/300
img_20 saved:     lat: 49.17104277 N, lon: -0.35114073 E        20/300
img_21 saved:     lat: 49.184190101 N, lon: -0.369936025 E        21/300
img_22 saved:     lat: 49.186889568 N, lon: -0.388961105 E        22/300
img_23 saved:     lat: 49.18495749 N, lon: -0.359720622 E        23/300
img_24 saved:     lat: 49.187185398 N, lon: -0.40386683 E        24/300
img_25 saved:     lat: 49.162902059 N, lon: -0.352227685 E        25/300
img_26 saved:     lat: 49.194515766 N, lon: -0.341853155 E        26/300
img_27 saved:     lat: 49.174920233 N, lon: -0.357244197 E        27/300
img_28 saved:     lat: 49.1801508 N, lon: -0.3582655 E        28/300
img_29 saved:     lat: 49.167674789 N, lon: -0.349916298 E        29/300
img_30 saved:     lat: 49.182771621 N, lon: -0.360170323 E        30/300
img_31 saved:     lat: 49.201409951 N, lon: -0.382131564 E        31/300
img_32 saved:     lat: 49.187320975 N, lon: -0.395796261 E        32/300
img_33 saved:     lat: 49.18528991 N, lon: -0.378810984 E        33/300
img_34 saved:     lat: 49.168135619 N, lon: -0.361614816 E        34/300
img_35 saved:     lat: 49.204081354 N, lon: -0.378124573 E        35/300
img_36 saved:     lat: 49.178317004 N, lon: -0.370944569 E        36/300
img_37 saved:     lat: 49.174062412 N, lon: -0.355809764 E        37/300
img_38 saved:     lat: 49.186611622 N, lon: -0.358356393 E        38/300
img_39 saved:     lat: 49.189159601 N, lon: -0.391489297 E        39/300
img_40 saved:     lat: 49.182447266 N, lon: -0.359229121 E        40/300
img_41 saved:     lat: 49.184687054 N, lon: -0.39403841 E        41/300
img_42 saved:     lat: 49.166138988 N, lon: -0.347055352 E        42/300
img_43 saved:     lat: 49.187774471 N, lon: -0.369905322 E        43/300
img_44 saved:     lat: 49.185431695 N, lon: -0.371605549 E        44/300
img_45 saved:     lat: 49.186623867 N, lon: -0.355695879 E        45/300
img_46 saved:     lat: 49.188694703 N, lon: -0.389540456 E        46/300
img_47 saved:     lat: 49.178243099 N, lon: -0.353424701 E        47/300
img_48 saved:     lat: 49.192584611 N, lon: -0.357248185 E        48/300
img_49 saved:     lat: 49.185292485 N, lon: -0.405842592 E        49/300
img_50 saved:     lat: 49.172424144 N, lon: -0.381144261 E        50/300
img_51 saved:     lat: 49.208956466 N, lon: -0.3813882 E        51/300
img_52 saved:     lat: 49.193481003 N, lon: -0.35669802 E        52/300
img_53 saved:     lat: 49.180649759 N, lon: -0.362265983 E        53/300
img_54 saved:     lat: 49.19295257 N, lon: -0.336025494 E        54/300
img_55 saved:     lat: 49.183280715 N, lon: -0.403342705 E        55/300
img_56 saved:     lat: 49.19898427 N, lon: -0.390624028 E        56/300
img_57 saved:     lat: 49.194697411 N, lon: -0.343656562 E        57/300
img_58 saved:     lat: 49.168876301 N, lon: -0.344135227 E        58/300
img_59 saved:     lat: 49.180513516 N, lon: -0.349972636 E        59/300
img_60 saved:     lat: 49.170710211 N, lon: -0.362462263 E        60/300
img_61 saved:     lat: 49.18253619 N, lon: -0.395209254 E        61/300
img_62 saved:     lat: 49.161311241 N, lon: -0.34218515 E        62/300
img_63 saved:     lat: 49.1731237 N, lon: -0.393589153 E        63/300
img_64 saved:     lat: 49.185578499 N, lon: -0.34273935 E        64/300
img_65 saved:     lat: 49.190267576 N, lon: -0.34089633 E        65/300
img_66 saved:     lat: 49.186572789 N, lon: -0.383089126 E        66/300
img_67 saved:     lat: 49.182181878 N, lon: -0.372195641 E        67/300
img_68 saved:     lat: 49.186756842 N, lon: -0.353597617 E        68/300
img_69 saved:     lat: 49.187052615 N, lon: -0.40290255 E        69/300
img_70 saved:     lat: 49.196328764 N, lon: -0.351418061 E        70/300
img_71 saved:     lat: 49.184904828 N, lon: -0.362842818 E        71/300
img_72 saved:     lat: 49.19190317 N, lon: -0.402411026 E        72/300
img_73 saved:     lat: 49.196776366 N, lon: -0.36408188 E        73/300
img_74 saved:     lat: 49.199382793 N, lon: -0.349160442 E        74/300
img_75 saved:     lat: 49.189582969 N, lon: -0.390609149 E        75/300
img_76 saved:     lat: 49.183673792 N, lon: -0.357040385 E        76/300
img_77 saved:     lat: 49.211262775 N, lon: -0.378421337 E        77/300
img_78 saved:     lat: 49.188224024 N, lon: -0.364137104 E        78/300
img_79 saved:     lat: 49.174878241 N, lon: -0.344775017 E        79/300
img_80 saved:     lat: 49.173783088 N, lon: -0.396970645 E        80/300
img_81 saved:     lat: 49.189113596 N, lon: -0.398719198 E        81/300
img_82 saved:     lat: 49.184135342 N, lon: -0.361576736 E        82/300
img_83 saved:     lat: 49.193645418 N, lon: -0.410130805 E        83/300
img_84 saved:     lat: 49.18913444 N, lon: -0.398759304 E        84/300
img_85 saved:     lat: 49.174751006 N, lon: -0.337589913 E        85/300
img_86 saved:     lat: 49.19898276 N, lon: -0.398795581 E        86/300
img_87 saved:     lat: 49.180726807 N, lon: -0.364103949 E        87/300
img_88 saved:     lat: 49.184117213 N, lon: -0.361588397 E        88/300
img_89 saved:     lat: 49.180586565 N, lon: -0.369305872 E        89/300
img_90 saved:     lat: 49.183409548 N, lon: -0.364876503 E        90/300
img_91 saved:     lat: 49.2056864 N, lon: -0.3791869 E        91/300
img_92 saved:     lat: 49.180600042 N, lon: -0.352693218 E        92/300
img_93 saved:     lat: 49.163310125 N, lon: -0.349090412 E        93/300
img_94 saved:     lat: 49.182158487 N, lon: -0.372133236 E        94/300
img_95 saved:     lat: 49.183458504 N, lon: -0.371264735 E        95/300
img_96 saved:     lat: 49.180649759 N, lon: -0.362265983 E        96/300
img_97 saved:     lat: 49.196782546 N, lon: -0.340483374 E        97/300
img_98 saved:     lat: 49.176651345 N, lon: -0.39369728 E        98/300
img_99 saved:     lat: 49.183358616 N, lon: -0.387007767 E        99/300
img_100 saved:     lat: 49.202187042 N, lon: -0.390162667 E        100/300
img_101 saved:     lat: 49.210290107 N, lon: -0.367589288 E        101/300
img_102 saved:     lat: 49.188262697 N, lon: -0.359830885 E        102/300
img_103 saved:     lat: 49.185079681 N, lon: -0.358535298 E        103/300
img_104 saved:     lat: 49.192588728 N, lon: -0.393506842 E        104/300
img_105 saved:     lat: 49.173103556 N, lon: -0.354278031 E        105/300
img_106 saved:     lat: 49.180998858 N, lon: -0.387999984 E        106/300
img_107 saved:     lat: 49.175588237 N, lon: -0.356535138 E        107/300
img_108 saved:     lat: 49.167544641 N, lon: -0.365461433 E        108/300
img_109 saved:     lat: 49.193510627 N, lon: -0.356717843 E        109/300
img_110 saved:     lat: 49.199114294 N, lon: -0.366343376 E        110/300
img_111 saved:     lat: 49.18558519 N, lon: -0.368783643 E        111/300
img_112 saved:     lat: 49.205848504 N, lon: -0.359390529 E        112/300
img_113 saved:     lat: 49.174848382 N, lon: -0.344873462 E        113/300
img_114 saved:     lat: 49.203572755 N, lon: -0.379199966 E        114/300
img_115 saved:     lat: 49.181321701 N, lon: -0.387835552 E        115/300
img_116 saved:     lat: 49.168161898 N, lon: -0.3884773 E        116/300
img_117 saved:     lat: 49.186022315 N, lon: -0.365463772 E        117/300
img_118 saved:     lat: 49.185290975 N, lon: -0.360028345 E        118/300
img_119 saved:     lat: 49.178242911 N, lon: -0.355676842 E        119/300
img_120 saved:     lat: 49.167347946 N, lon: -0.364684154 E        120/300
img_121 saved:     lat: 49.186092185 N, lon: -0.365345797 E        121/300
img_122 saved:     lat: 49.171994902 N, lon: -0.399384454 E        122/300
img_123 saved:     lat: 49.193242172 N, lon: -0.343069848 E        123/300
img_124 saved:     lat: 49.202142387 N, lon: -0.390132964 E        124/300
img_125 saved:     lat: 49.180820191 N, lon: -0.362311378 E        125/300
img_126 saved:     lat: 49.176348029 N, lon: -0.352926035 E        126/300
img_127 saved:     lat: 49.188102653 N, lon: -0.388087345 E        127/300
img_128 saved:     lat: 49.19255546 N, lon: -0.36765508 E        128/300
img_129 saved:     lat: 49.189245066 N, lon: -0.366381691 E        129/300
img_130 saved:     lat: 49.182746914 N, lon: -0.362003173 E        130/300
img_131 saved:     lat: 49.203722356 N, lon: -0.377016433 E        131/300
img_132 saved:     lat: 49.169658089 N, lon: -0.348323217 E        132/300
img_133 saved:     lat: 49.184546599 N, lon: -0.345160009 E        133/300
img_134 saved:     lat: 49.169790923 N, lon: -0.388646072 E        134/300
img_135 saved:     lat: 49.172413648 N, lon: -0.378513568 E        135/300
img_136 saved:     lat: 49.173363889 N, lon: -0.400956193 E        136/300
img_137 saved:     lat: 49.166510516 N, lon: -0.352035572 E        137/300
img_138 saved:     lat: 49.185949955 N, lon: -0.353333415 E        138/300
img_139 saved:     lat: 49.164305931 N, lon: -0.346782229 E        139/300
img_140 saved:     lat: 49.18192631 N, lon: -0.366950516 E        140/300
img_141 saved:     lat: 49.179793235 N, lon: -0.364135305 E        141/300
img_142 saved:     lat: 49.206350829 N, lon: -0.388159542 E        142/300
img_143 saved:     lat: 49.204524453 N, lon: -0.385254603 E        143/300
img_144 saved:     lat: 49.186798944 N, lon: -0.376570516 E        144/300
img_145 saved:     lat: 49.189036964 N, lon: -0.380871668 E        145/300
img_146 saved:     lat: 49.180489657 N, lon: -0.368951143 E        146/300
img_147 saved:     lat: 49.177919212 N, lon: -0.359934239 E        147/300
img_148 saved:     lat: 49.18277353 N, lon: -0.360183705 E        148/300
img_149 saved:     lat: 49.179861201 N, lon: -0.361019102 E        149/300
img_150 saved:     lat: 49.164442309 N, lon: -0.35914426 E        150/300
img_151 saved:     lat: 49.183373516 N, lon: -0.358835672 E        151/300
img_152 saved:     lat: 49.16527897 N, lon: -0.362940167 E        152/300
img_153 saved:     lat: 49.183911572 N, lon: -0.360045808 E        153/300
img_154 saved:     lat: 49.173185651 N, lon: -0.356489774 E        154/300
img_155 saved:     lat: 49.188067283 N, lon: -0.359931647 E        155/300
img_156 saved:     lat: 49.192198014 N, lon: -0.36865192 E        156/300
img_157 saved:     lat: 49.168025068 N, lon: -0.351700775 E        157/300
img_158 saved:     lat: 49.182373554 N, lon: -0.374635621 E        158/300
img_159 saved:     lat: 49.162008118 N, lon: -0.363489398 E        159/300
img_160 saved:     lat: 49.186323481 N, lon: -0.38976565 E        160/300
img_161 saved:     lat: 49.196570658 N, lon: -0.354190011 E        161/300
img_162 saved:     lat: 49.182447401 N, lon: -0.390487551 E        162/300
img_163 saved:     lat: 49.194095997 N, lon: -0.394832694 E        163/300
img_164 saved:     lat: 49.19308161 N, lon: -0.394571766 E        164/300
img_165 saved:     lat: 49.175248847 N, lon: -0.38711968 E        165/300
img_166 saved:     lat: 49.19112051 N, lon: -0.397485563 E        166/300
img_167 saved:     lat: 49.181282901 N, lon: -0.362742696 E        167/300
img_168 saved:     lat: 49.160846075 N, lon: -0.34765411 E        168/300
img_169 saved:     lat: 49.189531092 N, lon: -0.403603573 E        169/300
img_170 saved:     lat: 49.174840276 N, lon: -0.352511985 E        170/300
img_171 saved:     lat: 49.189486059 N, lon: -0.341276514 E        171/300
img_172 saved:     lat: 49.164859241 N, lon: -0.344649334 E        172/300
img_173 saved:     lat: 49.179098991 N, lon: -0.385971795 E        173/300
img_174 saved:     lat: 49.160147991 N, lon: -0.364687272 E        174/300
img_175 saved:     lat: 49.196776084 N, lon: -0.364116174 E        175/300
img_176 saved:     lat: 49.167324997 N, lon: -0.364719285 E        176/300
img_177 saved:     lat: 49.17285332 N, lon: -0.389358664 E        177/300
img_178 saved:     lat: 49.185470912 N, lon: -0.352805059 E        178/300
img_179 saved:     lat: 49.190118515 N, lon: -0.372007521 E        179/300
img_180 saved:     lat: 49.200526091 N, lon: -0.390731356 E        180/300
img_181 saved:     lat: 49.182569961 N, lon: -0.356630953 E        181/300
img_182 saved:     lat: 49.179013766 N, lon: -0.351853538 E        182/300
img_183 saved:     lat: 49.183444858 N, lon: -0.360846692 E        183/300
img_184 saved:     lat: 49.178478248 N, lon: -0.398694681 E        184/300
img_185 saved:     lat: 49.183917074 N, lon: -0.37574973 E        185/300
img_186 saved:     lat: 49.185625723 N, lon: -0.375118258 E        186/300
img_187 saved:     lat: 49.184977733 N, lon: -0.356288883 E        187/300
img_188 saved:     lat: 49.178780822 N, lon: -0.361254247 E        188/300
img_189 saved:     lat: 49.184047948 N, lon: -0.362838915 E        189/300
img_190 saved:     lat: 49.158247879 N, lon: -0.344666098 E        190/300
img_191 saved:     lat: 49.187851432 N, lon: -0.355695381 E        191/300
img_192 saved:     lat: 49.173370954 N, lon: -0.33668129 E        192/300
img_193 saved:     lat: 49.185075501 N, lon: -0.358066406 E        193/300
img_194 saved:     lat: 49.178941042 N, lon: -0.370807015 E        194/300
img_195 saved:     lat: 49.181719163 N, lon: -0.401651425 E        195/300
img_196 saved:     lat: 49.169679767 N, lon: -0.388606372 E        196/300
img_197 saved:     lat: 49.189487311 N, lon: -0.341279033 E        197/300
img_198 saved:     lat: 49.174377079 N, lon: -0.353772723 E        198/300
img_199 saved:     lat: 49.174116036 N, lon: -0.350970409 E        199/300
img_200 saved:     lat: 49.180912526 N, lon: -0.356265443 E        200/300
img_201 saved:     lat: 49.174014029 N, lon: -0.350587473 E        201/300
img_202 saved:     lat: 49.174234402 N, lon: -0.341613721 E        202/300
img_203 saved:     lat: 49.171361197 N, lon: -0.362853305 E        203/300
img_204 saved:     lat: 49.198193113 N, lon: -0.383109887 E        204/300
img_205 saved:     lat: 49.178895355 N, lon: -0.40244999 E        205/300
img_206 saved:     lat: 49.173136399 N, lon: -0.399516799 E        206/300
img_207 saved:     lat: 49.211059434 N, lon: -0.363401495 E        207/300
img_208 saved:     lat: 49.204198389 N, lon: -0.382920862 E        208/300
img_209 saved:     lat: 49.185519517 N, lon: -0.352326271 E        209/300
img_210 saved:     lat: 49.159333731 N, lon: -0.340674423 E        210/300
img_211 saved:     lat: 49.161336534 N, lon: -0.361704823 E        211/300
img_212 saved:     lat: 49.181463992 N, lon: -0.361225462 E        212/300
img_213 saved:     lat: 49.200959873 N, lon: -0.383861364 E        213/300
img_214 saved:     lat: 49.185621876 N, lon: -0.359739381 E        214/300
img_215 saved:     lat: 49.186387808 N, lon: -0.366702026 E        215/300
img_216 saved:     lat: 49.187557392 N, lon: -0.381650007 E        216/300
img_217 saved:     lat: 49.185961481 N, lon: -0.341374769 E        217/300
img_218 saved:     lat: 49.179294255 N, lon: -0.348988727 E        218/300
img_219 saved:     lat: 49.186996333 N, lon: -0.337298741 E        219/300
img_220 saved:     lat: 49.17217401 N, lon: -0.401976422 E        220/300
img_221 saved:     lat: 49.16100499 N, lon: -0.341709953 E        221/300
img_222 saved:     lat: 49.169224905 N, lon: -0.340395247 E        222/300
img_223 saved:     lat: 49.157918362 N, lon: -0.34197489 E        223/300
img_224 saved:     lat: 49.178319901 N, lon: -0.371062031 E        224/300
img_225 saved:     lat: 49.198298698 N, lon: -0.358742688 E        225/300
img_226 saved:     lat: 49.182416317 N, lon: -0.374443672 E        226/300
img_227 saved:     lat: 49.181214049 N, lon: -0.366070664 E        227/300
img_228 saved:     lat: 49.180349062 N, lon: -0.363080175 E        228/300
img_229 saved:     lat: 49.189945497 N, lon: -0.362561709 E        229/300
img_230 saved:     lat: 49.1882627 N, lon: -0.3598309 E        230/300
img_231 saved:     lat: 49.175996023 N, lon: -0.389352983 E        231/300
img_232 saved:     lat: 49.195149936 N, lon: -0.37775396 E        232/300
img_233 saved:     lat: 49.17226652 N, lon: -0.34062078 E        233/300
img_234 saved:     lat: 49.160157366 N, lon: -0.344132843 E        234/300
img_235 saved:     lat: 49.178503536 N, lon: -0.355800322 E        235/300
img_236 saved:     lat: 49.1767382 N, lon: -0.354662301 E        236/300
img_237 saved:     lat: 49.175567369 N, lon: -0.356670722 E        237/300
img_238 saved:     lat: 49.210992765 N, lon: -0.363213933 E        238/300
img_239 saved:     lat: 49.187806467 N, lon: -0.345417861 E        239/300
img_240 saved:     lat: 49.165563547 N, lon: -0.341510351 E        240/300
img_241 saved:     lat: 49.180826715 N, lon: -0.356430655 E        241/300
img_242 saved:     lat: 49.179906131 N, lon: -0.36104896 E        242/300
img_243 saved:     lat: 49.172874153 N, lon: -0.332407389 E        243/300
img_244 saved:     lat: 49.19110457 N, lon: -0.340439465 E        244/300
img_245 saved:     lat: 49.164266763 N, lon: -0.348098642 E        245/300
img_246 saved:     lat: 49.190785189 N, lon: -0.341882218 E        246/300
img_247 saved:     lat: 49.178549099 N, lon: -0.36146529 E        247/300
img_248 saved:     lat: 49.181415364 N, lon: -0.362851787 E        248/300
img_249 saved:     lat: 49.200427737 N, lon: -0.3633531 E        249/300
img_250 saved:     lat: 49.179029945 N, lon: -0.354229107 E        250/300
img_251 saved:     lat: 49.195966902 N, lon: -0.367946093 E        251/300
img_252 saved:     lat: 49.1626622 N, lon: -0.346241421 E        252/300
img_253 saved:     lat: 49.179307268 N, lon: -0.348981858 E        253/300
img_254 saved:     lat: 49.171563488 N, lon: -0.349265291 E        254/300
img_255 saved:     lat: 49.18344551 N, lon: -0.360855314 E        255/300
img_256 saved:     lat: 49.197896142 N, lon: -0.352296985 E        256/300
img_257 saved:     lat: 49.184621677 N, lon: -0.364798031 E        257/300
img_258 saved:     lat: 49.185047542 N, lon: -0.365742532 E        258/300
img_259 saved:     lat: 49.165888661 N, lon: -0.362748223 E        259/300
img_260 saved:     lat: 49.188714301 N, lon: -0.402328149 E        260/300
img_261 saved:     lat: 49.183610554 N, lon: -0.384948229 E        261/300
img_262 saved:     lat: 49.189151424 N, lon: -0.391477125 E        262/300
img_263 saved:     lat: 49.171918357 N, lon: -0.334509298 E        263/300
img_264 saved:     lat: 49.188637759 N, lon: -0.380338386 E        264/300
img_265 saved:     lat: 49.187816546 N, lon: -0.3668558 E        265/300
img_266 saved:     lat: 49.199281912 N, lon: -0.364183519 E        266/300
img_267 saved:     lat: 49.187555052 N, lon: -0.393446518 E        267/300
img_268 saved:     lat: 49.190203467 N, lon: -0.400311768 E        268/300
img_269 saved:     lat: 49.183140971 N, lon: -0.35262342 E        269/300
img_270 saved:     lat: 49.182571112 N, lon: -0.35483227 E        270/300
img_271 saved:     lat: 49.171854759 N, lon: -0.345848692 E        271/300
img_272 saved:     lat: 49.179014844 N, lon: -0.354261562 E        272/300
img_273 saved:     lat: 49.199284584 N, lon: -0.364178502 E        273/300
img_274 saved:     lat: 49.19215729 N, lon: -0.403265652 E        274/300
img_275 saved:     lat: 49.174845473 N, lon: -0.352496413 E        275/300
img_276 saved:     lat: 49.182991557 N, lon: -0.332337689 E        276/300
img_277 saved:     lat: 49.162009401 N, lon: -0.36350643 E        277/300
img_278 saved:     lat: 49.162010172 N, lon: -0.363471592 E        278/300
img_279 saved:     lat: 49.180208793 N, lon: -0.350106333 E        279/300
img_280 saved:     lat: 49.174622168 N, lon: -0.377870237 E        280/300
img_281 saved:     lat: 49.171871824 N, lon: -0.377111852 E        281/300
img_282 saved:     lat: 49.185040006 N, lon: -0.352207755 E        282/300
img_283 saved:     lat: 49.161504346 N, lon: -0.365067576 E        283/300
img_284 saved:     lat: 49.210129096 N, lon: -0.378457182 E        284/300
img_285 saved:     lat: 49.186959332 N, lon: -0.402624191 E        285/300
img_286 saved:     lat: 49.193119132 N, lon: -0.379318006 E        286/300
img_287 saved:     lat: 49.192471064 N, lon: -0.399775688 E        287/300
img_288 saved:     lat: 49.190613031 N, lon: -0.346982304 E        288/300
img_289 saved:     lat: 49.158528928 N, lon: -0.337824898 E        289/300
img_290 saved:     lat: 49.178459132 N, lon: -0.401295325 E        290/300
img_291 saved:     lat: 49.174648845 N, lon: -0.39152628 E        291/300
img_292 saved:     lat: 49.210983098 N, lon: -0.363222211 E        292/300
img_293 saved:     lat: 49.172411804 N, lon: -0.350325092 E        293/300
img_294 saved:     lat: 49.172114955 N, lon: -0.335435497 E        294/300
img_295 saved:     lat: 49.184408498 N, lon: -0.35913705 E        295/300
img_296 saved:     lat: 49.183138311 N, lon: -0.386877831 E        296/300
img_297 saved:     lat: 49.180158211 N, lon: -0.349998694 E        297/300
img_298 saved:     lat: 49.198454799 N, lon: -0.3618146 E        298/300
img_299 saved:     lat: 49.173757692 N, lon: -0.341397053 E        299/300
img_300 saved:     lat: 49.169415999 N, lon: -0.361898568 E        300/300